# Regression vs Classification Strategies: Linear, Lasso, Logistic & Decision Trees

Every model sees the same 1/5/15/20-day momentum features of S&P 500 stocks, on the same date-based train/test split. The **regressions** forecast the next-day *return*; the **classifiers** forecast the *probability* it is positive. Each model becomes a long/flat strategy, and **all of them are compared against the equal-weighted benchmark in one Plotly dashboard**.

| Model | Predicts | Long when |
|---|---|---|
| Linear regression (OLS) | next-day return | forecast > 0 |
| Lasso (alpha by time-series CV) | next-day return | forecast > 0 |
| Logistic regression | P(next-day return > 0) | P > 0.5 |
| Decision tree (default) | P(up) | P > 0.5 |
| Decision tree (tuned with time-ordered CV) | P(up) | P > 0.5 |

**Everything shared comes from `func_lib.py`:**
- Data: `createHistPrices`, `computingReturns`
- Split, scaling and CV folds: `chronological_split`, `scale_features`, `date_based_cv_splits`
- Backtests: `build_pred_frame`, `compute_BM_Perf`, `compute_strat_perf` (with `plot=False`)
- Comparison: `strategy_comparison_figure`, `strategy_stats_table`, `stats_table_figure`, coefficient charts
- The dark Plotly theme

In [1]:
# Imports: statsmodels for OLS/logit with full statistics, scikit-learn for Lasso and trees,
# func_lib for the data, split, backtest and chart helpers
import os
import sys

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import Lasso, LassoCV
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

# Make func_lib importable from this folder or its parent, without duplicate sys.path entries
for path in (os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), '..'))):
    if os.path.exists(os.path.join(path, 'func_lib.py')) and path not in sys.path:
        sys.path.append(path)

import func_lib   # importing it also applies the dark Plotly theme ('quant_dark')

In [2]:
# Configuration: date range, momentum feature windows, train/test split, long/flat thresholds, Lasso alphas
START_DATE, END_DATE = '2000-01-01', '2026-09-23'
MOMENTUM_WINDOWS = [1, 5, 15, 20]
TRAIN_FRACTION = 0.7       # first 70% of dates for training, rest for testing
PROB_THRESHOLD = 0.5       # classifiers: long when P(up) > threshold
RETURN_THRESHOLD = 0.0     # regressions: long when forecast return > threshold
CV_SPLITS = 5
RANDOM_STATE = 42

USE_LASSO_CV = True        # choose Lasso alpha by time-series CV
FIXED_ALPHA = 1e-5         # used when USE_LASSO_CV is False
ALPHA_GRID = np.logspace(-7, -2, 30)

FEATURES = [f'{m}_d_returns' for m in MOMENTUM_WINDOWS]
RETURN_COL = 'F_1_d_returns'         # regression target and realized forward return
DIRECTION_COL = 'F_1_d_returns_Ind'  # classification target: 1 if forward return > 0

## 1. Data and one shared chronological split

In [3]:
# Download S&P 500 prices, build momentum features and the next-day return, and make one chronological
# split for every model. Features are scaled on training data only; CV folds are cut by date (no look-ahead).
historical_prices = func_lib.createHistPrices(START_DATE, END_DATE)
total_returns = func_lib.computingReturns(historical_prices, MOMENTUM_WINDOWS)
total_returns[DIRECTION_COL] = (total_returns[RETURN_COL] > 0).astype(int)

split = func_lib.chronological_split(total_returns, FEATURES, RETURN_COL, RETURN_COL, TRAIN_FRACTION)
x_train, x_test = split['x_train'], split['x_test']
y_ret_train, y_ret_test = split['y_train'], split['y_test']                 # regression targets
y_dir_train = total_returns.loc[x_train.index, DIRECTION_COL]               # classification targets
y_dir_test = total_returns.loc[x_test.index, DIRECTION_COL]
test_fwd_returns = split['fwd_returns_test']
x_train_scaled, x_test_scaled, scaler = func_lib.scale_features(x_train, x_test)
cv_folds = func_lib.date_based_cv_splits(x_train_scaled.index, CV_SPLITS)

print(f"Split date: {split['split_date']} | train rows: {len(x_train):,} | test rows: {len(x_test):,}")
print(f'Share of up days  train: {y_dir_train.mean():.3f} | test: {y_dir_test.mean():.3f}')
total_returns.head()

$VYLR: Data doesn't exist for startDate = 946702800, endDate = 1790136000


$SPG: possibly delisted; no price data found  (1d 2000-01-01 -> 2026-09-23)



2 Failed downloads:


['VYLR']: Data doesn't exist for startDate = 946702800, endDate = 1790136000


['SPG']: possibly delisted; no price data found  (1d 2000-01-01 -> 2026-09-23)


Split date: 2018-09-20 00:00:00 | train rows: 1,930,108 | test rows: 981,781
Share of up days  train: 0.511 | test: 0.520


F_1_d_returns  1_d_returns  5_d_returns  15_d_returns  \
Ticker Date                                                                
A      2000-02-01       0.077465     0.072710     0.048938      0.029919   
       2000-02-02       0.016340     0.077465     0.119854      0.125000   
       2000-02-03      -0.019293     0.016340     0.136073      0.166980   
       2000-02-04       0.042623    -0.019293     0.120294      0.127541   
       2000-02-07      -0.006289     0.042623     0.201133      0.162706   

                   20_d_returns  F_1_d_returns_Ind  
Ticker Date                                         
A      2000-02-01     -0.013889                  1  
       2000-02-02      0.150376                  1  
       2000-02-03      0.246493                  0  
       2000-02-04      0.270833                  1  
       2000-02-07      0.223077                  0

## 2. Regression models: forecast the next-day return

In [4]:
# Linear regression (OLS) of next-day return on the scaled features; statsmodels gives the
# coefficient table with standard errors and p-values
x_train_const = sm.add_constant(x_train_scaled)
x_test_const = sm.add_constant(x_test_scaled)

ols_model = sm.OLS(y_ret_train, x_train_const).fit()
print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:          F_1_d_returns   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     674.0
Date:                Mon, 09 Nov 2026   Prob (F-statistic):               0.00
Time:                        20:39:35   Log-Likelihood:             4.4929e+06
No. Observations:             1930108   AIC:                        -8.986e+06
Df Residuals:                 1930103   BIC:                        -8.986e+06
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.0007    1.7e-05     43.453   

In [5]:
# Lasso: OLS plus an L1 penalty that shrinks weak coefficients to exactly zero, with alpha chosen
# on date-ordered folds. Then compare the OLS and Lasso coefficients side by side.
if USE_LASSO_CV:
    lasso_model = LassoCV(alphas=ALPHA_GRID, cv=cv_folds, n_jobs=-1).fit(x_train_scaled, y_ret_train)
    print(f'CV-selected alpha: {lasso_model.alpha_:.2e}')
else:
    lasso_model = Lasso(alpha=FIXED_ALPHA).fit(x_train_scaled, y_ret_train)

coefficients = pd.DataFrame({'Linear (OLS)': ols_model.params[FEATURES],
                             'Lasso': pd.Series(lasso_model.coef_, index=FEATURES)})
print(coefficients.assign(lasso_selected=coefficients['Lasso'] != 0))
func_lib.coefficient_comparison_figure(coefficients, 'Linear vs Lasso coefficients (scaled features)').show()

CV-selected alpha: 1.27e-04
              Linear (OLS)     Lasso  lasso_selected
1_d_returns      -0.000121 -0.000032            True
5_d_returns      -0.000641 -0.000593            True
15_d_returns     -0.000499 -0.000180            True
20_d_returns      0.000286 -0.000000           False


## 3. Classification models: forecast P(next-day return > 0)

In [6]:
# Logistic regression: P(tomorrow's return > 0). Coefficient table, then a chart of the coefficients (log-odds)
logit_model = sm.Logit(y_dir_train, x_train_const).fit(disp=False)
print(logit_model.summary())
func_lib.coefficient_bar_figure(logit_model.params[FEATURES], 'Logistic regression coefficients (log-odds)').show()

                           Logit Regression Results                           
Dep. Variable:      F_1_d_returns_Ind   No. Observations:              1930108
Model:                          Logit   Df Residuals:                  1930103
Method:                           MLE   Df Model:                            4
Date:                Mon, 09 Nov 2026   Pseudo R-squ.:               0.0005983
Time:                        20:39:38   Log-Likelihood:            -1.3366e+06
converged:                       True   LL-Null:                   -1.3374e+06
Covariance Type:            nonrobust   LLR p-value:                     0.000
                   coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.0423      0.001     29.339      0.000       0.039       0.045
1_d_returns     -0.0299      0.002    -18.623      0.000      -0.033      -0.027
5_d_returns     -0.0252      0.002    -13.52

In [7]:
# Default tree: an unpruned tree overfits noisy daily returns, so it serves as a baseline
tree_model = DecisionTreeClassifier(random_state=RANDOM_STATE).fit(x_train_scaled, y_dir_train)

# Tuned tree: depth and leaf sizes chosen by grid search on date-ordered expanding-window folds
grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid={'criterion': ['gini'], 'max_depth': [None, 5, 10, 15],
                'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4]},
    cv=cv_folds, scoring='roc_auc', n_jobs=-1,
).fit(x_train_scaled, y_dir_train)
print('Best params:', grid_search.best_params_, f'| CV ROC-AUC: {grid_search.best_score_:.4f}')

/Users/damionm_/miniforge3/envs/quantdev/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best params: {'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2} | CV ROC-AUC: 0.5165


## 4. Out-of-sample predictions and predictive accuracy
Each model produces a score: a forecast return or a probability.
- **Hit rate:** how often the long/flat call matched the actual direction.
- **ROC-AUC:** how well the score *ranks* up days above down days (0.5 = no skill).
- **Rank IC:** the Spearman correlation between the score and the realized return.

In [8]:
# Out-of-sample score of every model on the test period: hit rate, ROC-AUC, rank IC and time long
MODELS = {   # name -> (out-of-sample score, kind)
    'Linear':    (ols_model.predict(x_test_const), 'regression'),
    'Lasso':     (lasso_model.predict(x_test_scaled), 'regression'),
    'Logistic':  (logit_model.predict(x_test_const), 'classifier'),
    'DecTree':   (tree_model.predict_proba(x_test_scaled)[:, 1], 'classifier'),
    'DecTreeCV': (grid_search.best_estimator_.predict_proba(x_test_scaled)[:, 1], 'classifier'),
}
THRESHOLDS = {'regression': RETURN_THRESHOLD, 'classifier': PROB_THRESHOLD}

accuracy = {}
for name, (score, kind) in MODELS.items():
    score = pd.Series(np.asarray(score).ravel(), index=x_test.index)
    long = score > THRESHOLDS[kind]
    # Rank IC is undefined for a constant score (e.g. a Lasso that zeroed every coefficient)
    rank_ic = score.corr(test_fwd_returns, method='spearman') if score.nunique() > 1 else np.nan
    accuracy[name] = {'Type': kind.title(), 'Hit Rate': accuracy_score(y_dir_test, long),
                      'ROC-AUC': roc_auc_score(y_dir_test, score), 'Rank IC': rank_ic, 'Time Long': long.mean()}
accuracy = pd.DataFrame(accuracy)
func_lib.stats_table_figure(accuracy, 'Out-of-sample predictive accuracy', index_name='Metric',
                            pct_rows=('Hit Rate', 'Time Long'), decimals=4).show()

## 5. Backtest every strategy against the benchmark
Each strategy is equal-weighted across the stocks it is long each day, and flat otherwise. The benchmark is the equal-weighted average of all stocks. `plot=False` skips the per-model charts, so one dashboard compares everything.

In [9]:
# Backtest: each model becomes a long/flat strategy (position rule from THRESHOLDS above),
# compared with the equal-weighted benchmark
def long_flat_rule(kind):
    """Position rule for a model kind: long (1) when the score beats its threshold, otherwise flat (0)."""
    threshold = THRESHOLDS[kind]
    return lambda score: int(score > threshold)


# The benchmark only uses the realized returns, so any model's prediction frame works as its input
first_name, (first_score, _) = next(iter(MODELS.items()))
cum_returns, calendar_returns = func_lib.compute_BM_Perf(
    func_lib.build_pred_frame(first_score, x_test.index, test_fwd_returns, first_name), plot=False)

for name, (score, kind) in MODELS.items():
    preds = func_lib.build_pred_frame(score, x_test.index, test_fwd_returns, name)
    cum_returns, calendar_returns = func_lib.compute_strat_perf(
        preds, cum_returns, calendar_returns, long_flat_rule(kind), name, plot=False)

 The Model is: S&P500 (equal-weighted benchmark)
 The CAGR is: 16.81%
 Sharpe Ratio of Strategy: 0.87


 The Model is: Linear
 The CAGR is: 15.33%
 Sharpe Ratio of Strategy: 0.88
 The Model is: Lasso
 The CAGR is: 16.03%
 Sharpe Ratio of Strategy: 0.89


 The Model is: Logistic
 The CAGR is: 15.47%
 Sharpe Ratio of Strategy: 0.91
 The Model is: DecTree
 The CAGR is: 9.09%
 Sharpe Ratio of Strategy: 0.90


 The Model is: DecTreeCV
 The CAGR is: 14.49%
 Sharpe Ratio of Strategy: 1.01


In [10]:
# Every strategy against the benchmark: growth of $1, drawdowns and calendar-year returns
func_lib.strategy_comparison_figure(
    cum_returns, calendar_returns,
    func_lib.subtitled('All strategies vs benchmark', f"Equal-weighted benchmark, test period from {split['split_date']:%Y-%m-%d}"),
).show()

In [11]:
# Performance table for every strategy and the benchmark, then the strategy with the highest Sharpe ratio
comparison = func_lib.strategy_stats_table(cum_returns)
func_lib.stats_table_figure(
    comparison, 'Strategy performance vs benchmark (out-of-sample)', index_name='Metric',
    pct_rows=('Total Return', 'CAGR', 'Annualized Return (arith.)', 'Annualized Volatility',
              'Downside Deviation', 'Max Drawdown', 'Positive Days', 'Excess CAGR vs Benchmark'),
).show()

BENCHMARK = 'S&P500'   # column name compute_BM_Perf gives the benchmark
sharpe = comparison.loc['Sharpe Ratio']
best = sharpe.drop(BENCHMARK).idxmax()
print(f'Highest Sharpe: {best} ({sharpe[best]:.2f}) vs benchmark {sharpe[BENCHMARK]:.2f}')

Highest Sharpe: DecTreeCV (1.01) vs benchmark 0.87


## 6. Reading the results
- **Regression vs classification:** both families use the same information, so differences mostly come from the loss function. A model that ranks days well (higher AUC / IC) can still lose to the benchmark if it sits flat during strong up-markets. Check **Time Long** next to **Excess CAGR**.
- **Lasso shrinkage:** if CV shrinks every coefficient to zero, the Lasso forecast is a constant, and the strategy is always long (equal to the benchmark) or always flat.
- **Trees:** the unpruned tree is the overfitting baseline. The tuned tree uses date-ordered CV, so its improvement (if any) is honest out-of-sample.
- **Next steps:** add transaction costs on position changes, try a cross-sectional ranking (long the top decile instead of long/flat), and add gradient boosting (Bali et al. found nonlinear models help most).